In [1]:
# %% [markdown]
# # Notebook 1: Data Cleaning, Circular Wind Aggregation, and Merging

# %%
import os
import numpy as np
import pandas as pd

# 1. Resolve file paths (supports running from root or notebooks/ folder)
poll_paths = ["../data/raw/air_poll_ITO_(2018-2025).csv", "data/raw/air_poll_ITO_(2018-2025).csv", "air_poll_ITO_(2018-2025).csv"]
weather_paths = ["../data/raw/weather_dataset_delhi_2020_2026.csv", "data/raw/weather_dataset_delhi_2020_2026.csv", "weather_dataset_delhi_2020_2026.csv"]

raw_poll_file = next(p for p in poll_paths if os.path.exists(p))
raw_weather_file = next(w for w in weather_paths if os.path.exists(w))

print(f"Loading pollution from: {raw_poll_file}")
print(f"Loading weather from:   {raw_weather_file}")

pollution = pd.read_csv(raw_poll_file)
weather = pd.read_csv(raw_weather_file)

# %%
# 2. Clean pollution column names & parse day-first timestamps
pollution.columns = pollution.columns.str.strip()
pollution["Date"] = pd.to_datetime(pollution["Day"], format="%d-%m-%Y", errors="coerce")
pollution = pollution.dropna(subset=["Date"])

# Restrict pollution to the 2020-2025 analysis window
pollution = pollution[(pollution["Date"] >= "2020-01-01") & (pollution["Date"] <= "2025-12-31")].copy()
pollution = pollution[["Date", "PM2.5", "PM10", "NO2", "SO2"]].sort_values("Date").reset_index(drop=True)

print(f"Pollution records (2020-2025): {len(pollution)}")

# %%
# 3. Clean hourly weather timestamps & filter window
weather["time"] = pd.to_datetime(weather["time"])
weather["Date"] = weather["time"].dt.normalize()
weather = weather[(weather["Date"] >= "2020-01-01") & (weather["Date"] <= "2025-12-31")].copy()

# Audit hourly completeness
hours_per_day = weather.groupby("Date").size()
print(f"Days with 24 hours: {(hours_per_day == 24).sum()} / {len(hours_per_day)}")

# %%
# 4. Circular wind direction handling via vector components:
# u = speed * sin(theta), v = speed * cos(theta)
theta = np.deg2rad(weather["wind_direction"])
weather["wind_u"] = weather["wind_speed"] * np.sin(theta)
weather["wind_v"] = weather["wind_speed"] * np.cos(theta)

weather_season = weather.groupby("Date")["season"].first().reset_index()

# %%
# 5. Aggregate hourly weather to daily metrics
weather_daily = weather.groupby("Date").agg(
    temperature_mean=("temperature", "mean"),
    temperature_min=("temperature", "min"),
    temperature_max=("temperature", "max"),
    humidity_mean=("humidity", "mean"),
    dew_point_mean=("dew_point", "mean"),
    feels_like_mean=("feels_like", "mean"),
    rain_sum=("rain", "sum"),
    pressure_mean=("pressure", "mean"),
    cloud_cover_mean=("cloud_cover", "mean"),
    wind_speed_mean=("wind_speed", "mean"),
    wind_speed_max=("wind_speed", "max"),
    wind_u_mean=("wind_u", "mean"),
    wind_v_mean=("wind_v", "mean")
).reset_index()

# Reconstruct circular vector wind metrics
weather_daily["wind_speed_vector"] = np.sqrt(weather_daily["wind_u_mean"]**2 + weather_daily["wind_v_mean"]**2)
weather_daily["wind_direction_vector"] = (np.degrees(np.arctan2(weather_daily["wind_u_mean"], weather_daily["wind_v_mean"])) % 360)
weather_daily = weather_daily.merge(weather_season, on="Date", how="left")

# %%
# 6. Merge pollution and daily weather
df = pollution.merge(weather_daily, on="Date", how="inner").sort_values("Date").reset_index(drop=True)

# 7. Add calendar indicators
df["year"] = df["Date"].dt.year
df["month"] = df["Date"].dt.month
df["day_of_week"] = df["Date"].dt.dayofweek
df["quarter"] = df["Date"].dt.quarter
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

print(f"Final merged dataframe shape: {df.shape}")
print(f"Calendar span: {df['Date'].min().strftime('%Y-%m-%d')} to {df['Date'].max().strftime('%Y-%m-%d')}")
print(f"Missing values by column:\n{df.isna().sum()[df.isna().sum() > 0]}")

# %%
# 8. Save master daily processed dataset
out_dirs = ["../data/processed", "data/processed", "."]
target_dir = next((d for d in out_dirs if os.path.exists(d)), "../data/processed")
os.makedirs(target_dir, exist_ok=True)
out_file = os.path.join(target_dir, "delhi_pollution_weather_daily_2020_2025.csv")

df.to_csv(out_file, index=False)
print(f"Dataset successfully exported to: {out_file}")

Loading pollution from: ../data/raw/air_poll_ITO_(2018-2025).csv
Loading weather from:   ../data/raw/weather_dataset_delhi_2020_2026.csv
Pollution records (2020-2025): 2192
Days with 24 hours: 2192 / 2192
Final merged dataframe shape: (2192, 26)
Calendar span: 2020-01-01 to 2025-12-31
Missing values by column:
PM2.5     62
PM10     207
NO2       53
SO2       28
dtype: int64
Dataset successfully exported to: ../data/processed\delhi_pollution_weather_daily_2020_2025.csv
